In [8]:
import pandas as pd
import numpy as np
import joblib
import random

# 1. Load Models and Data for the book_recommender project
log_model = joblib.load('book_rec_logistic_model.pkl')
scaler = joblib.load('book_rec_scaler.pkl')
xgb_ranker = joblib.load('book_rec_xgboost_ranker.pkl')

df_matrix = pd.read_csv('book_recommender_master_feature_matrix_v2.csv')

# Dynamically select the correct author column from the unbiased baseline
df_books_raw = pd.read_csv('unbiased_book_recommender_50k.csv')
author_col = next((c for c in ['author_id', 'author_ids', 'authors'] if c in df_books_raw.columns), None)
cols_to_keep = ['book_id', 'title', 'num_pages', 'genres']
if author_col:
    cols_to_keep.append(author_col)

df_books = df_books_raw[cols_to_keep].drop_duplicates(subset=['book_id'])

# 2. Select a Random Active User (Must have at least 10 historical interactions)
user_counts = df_matrix['user_id'].value_counts()
active_users = user_counts[user_counts >= 10].index.tolist()
target_user = random.choice(active_users)

print(f"Generating book recommendations for User: {target_user}...\n")

# 3. Generate Candidate Books (Books the user hasn't seen)
user_history = df_matrix[df_matrix['user_id'] == target_user]
read_book_ids = user_history['book_id'].unique()
candidates = df_books[~df_books['book_id'].isin(read_book_ids)].copy()

# For speed in this execution, take a random sample of 500 candidate books
candidates = candidates.sample(n=min(500, len(candidates)), random_state=42).copy()

# 4. Construct the Feature Matrix for Candidates based on User's Latest Habits
latest_user_profile = user_history.iloc[-1]
overall_conv = latest_user_profile['overall_shelf_to_read_conversion']

candidates['overall_shelf_to_read_conversion'] = overall_conv
candidates['is_short'] = ((candidates['num_pages'] > 0) & (candidates['num_pages'] < 200)).astype(int)
candidates['is_medium'] = ((candidates['num_pages'] >= 200) & (candidates['num_pages'] <= 400)).astype(int)
candidates['is_long'] = (candidates['num_pages'] > 400).astype(int)

candidates['short_conversion_rate'] = np.where(candidates['is_short'] == 1, latest_user_profile['short_conversion_rate'], 0)
candidates['medium_conversion_rate'] = np.where(candidates['is_medium'] == 1, latest_user_profile['medium_conversion_rate'], 0)
candidates['long_conversion_rate'] = np.where(candidates['is_long'] == 1, latest_user_profile['long_conversion_rate'], 0)

# 5. Fallback approximations for unobserved Genre and Author combinations
candidates['genre_conversion_rate'] = 0.0  
candidates['genre_preference'] = 0.5       
candidates['author_conversion_rate'] = 0.0

feature_cols = [
    'overall_shelf_to_read_conversion', 'short_conversion_rate', 
    'medium_conversion_rate', 'long_conversion_rate',
    'genre_conversion_rate', 'genre_preference', 'author_conversion_rate'
]

# 6. Predict Conversion Probabilities (Step 3 Logistic Regression Model)
X_candidates_scaled = scaler.transform(candidates[feature_cols])

# FIXED: Exact column name expected by XGBoost
candidates['conversion_probability'] = log_model.predict_proba(X_candidates_scaled)[:, 1]

# 7. Final Ranking (Step 4 XGBoost Model)
candidates['preference_score'] = candidates['genre_preference']
X_rank = candidates[['preference_score', 'conversion_probability']]
candidates['final_rank_score'] = xgb_ranker.predict_proba(X_rank)[:, 1]

# 8. Sort and Display Top 3 Recommendations with Explanations
top_recs = candidates.sort_values(by='final_rank_score', ascending=False).head(3)

print("=== TOP 3 PERSONALIZED RECOMMENDATIONS ===\n")
for i, (_, row) in enumerate(top_recs.iterrows(), 1):
    print(f"#{i}: {row['title']}")
    print(f"    - Genre: {row['genres']}")
    print(f"    - Pages: {row['num_pages']}")
    print(f"    - Conversion Probability: {row['conversion_probability'] * 100:.1f}%")
    
    # Rule-Based Explanation Generation
    length_str = "short" if row['is_short'] else "medium" if row['is_medium'] else "long"
    print(f"    - WHY THIS BOOK? Recommended because you have historically converted {length_str} books at a steady rate, and your overall reading activity indicates a high probability ({row['conversion_probability'] * 100:.1f}%) of finishing this title within 90 days.\n")

Generating book recommendations for User: d33e59b212ac82591492dc3a47284161...

=== TOP 3 PERSONALIZED RECOMMENDATIONS ===

#1: Beyond Religion: Ethics for a Whole World
    - Genre: ['to-read' 'non-fiction' 'currently-reading' 'religion' 'philosophy'
 'buddhism' 'spirituality' 'nonfiction' 'spiritual' 'ethics' 'favorites'
 'audiobooks' 'audio' 'self-help' 'audiobook' 'audible'
 'religion-philosophy' 'psychology' 'owned' 'library' 'dalai-lama'
 'to-buy' 'tibet' 'inspirational' 'to-read-non-fiction' 'ebook'
 'books-i-own' 'science' 'kindle' 'spirit' 'religion-spirituality'
 'religious' 'religious-studies' 'self-improvement' 'mindfulness'
 'audio-book' 'partially-read' 'inspiration' 'buddhist'
 'philosophy-religion' 'religion-and-spirituality' 'wish-list'
 'self-development' 'life' 'politics' 'audio-books' 'paused' 'unfinished'
 'faith' 'listened' 'dalai-lama-xiv' 'values' 'secularism' 'to-listen'
 'nook' 'spirituality-religion' 'far-east-philosophy' 'own-but-not-read'
 'tai-lopez' 'stral

To make this a realistic, production-ready recommender for the looknew project, we cannot rely on the 0.0 fallbacks. We need to implement a matching function that compares the candidate book's actual author and genres against the user's real historical metrics matrix, ensuring a short fantasy book scores differently than a short mystery book.

In [9]:
import pandas as pd
import numpy as np
import joblib
import random

# 1. Load Models and Data for the book_recommender project
log_model = joblib.load('book_rec_logistic_model.pkl')
scaler = joblib.load('book_rec_scaler.pkl')
xgb_ranker = joblib.load('book_rec_xgboost_ranker.pkl')

df_matrix = pd.read_csv('book_recommender_master_feature_matrix_v2.csv')

# Load raw books and aggressively standardize the author column
df_books_raw = pd.read_csv('unbiased_book_recommender_50k.csv')

# Find whatever author column exists and rename it immediately
author_col = next((c for c in ['author_id', 'author_ids', 'authors'] if c in df_books_raw.columns), None)
if author_col and author_col != 'author_id':
    df_books_raw.rename(columns={author_col: 'author_id'}, inplace=True)

# Safety fallback: if no author column exists at all, create a blank one
if 'author_id' not in df_books_raw.columns:
    df_books_raw['author_id'] = 'unknown'

# Now safely slice using the guaranteed column names
cols_to_keep = ['book_id', 'title', 'num_pages', 'genres', 'author_id']
df_books = df_books_raw[cols_to_keep].drop_duplicates(subset=['book_id']).copy()

df_books['book_id'] = df_books['book_id'].astype(str)
df_matrix['book_id'] = df_matrix['book_id'].astype(str)

# 2. Select a Random Active User (Must have at least 10 interactions)
user_counts = df_matrix['user_id'].value_counts()
active_users = user_counts[user_counts >= 10].index.tolist()
target_user = random.choice(active_users)

print(f"Generating optimized book recommendations for User: {target_user}...\n")

# 3. Generate Candidate Books (Books the user hasn't seen)
user_history = df_matrix[df_matrix['user_id'] == target_user].copy()
read_book_ids = user_history['book_id'].unique()
candidates = df_books[~df_books['book_id'].isin(read_book_ids)].copy()

# For speed in this demo, take a random sample of 1000 candidate books
candidates = candidates.sample(n=min(1000, len(candidates)), random_state=42).copy()

# =======================================================
# 4. DYNAMIC PREFERENCE MATCHING
# =======================================================
# Merge history with book data to see WHICH genres/authors the user read
user_history_full = pd.merge(user_history, df_books[['book_id', 'genres', 'author_id']], on='book_id', how='left')

# Extract the user's latest conversion rates per genre
latest_genre_stats = user_history_full.drop_duplicates(subset=['genres'], keep='last')
genre_conv_map = dict(zip(latest_genre_stats['genres'], latest_genre_stats['genre_conversion_rate']))
genre_pref_map = dict(zip(latest_genre_stats['genres'], latest_genre_stats['genre_preference']))

# Extract the user's latest conversion rates per author
latest_author_stats = user_history_full.drop_duplicates(subset=['author_id'], keep='last')
author_conv_map = dict(zip(latest_author_stats['author_id'], latest_author_stats['author_conversion_rate']))

# =======================================================
# 5. CONSTRUCT CANDIDATE FEATURE MATRIX
# =======================================================
latest_user_profile = user_history.iloc[-1]

candidates['overall_shelf_to_read_conversion'] = latest_user_profile['overall_shelf_to_read_conversion']
candidates['is_short'] = ((candidates['num_pages'] > 0) & (candidates['num_pages'] < 200)).astype(int)
candidates['is_medium'] = ((candidates['num_pages'] >= 200) & (candidates['num_pages'] <= 400)).astype(int)
candidates['is_long'] = (candidates['num_pages'] > 400).astype(int)

candidates['short_conversion_rate'] = np.where(candidates['is_short'] == 1, latest_user_profile['short_conversion_rate'], 0)
candidates['medium_conversion_rate'] = np.where(candidates['is_medium'] == 1, latest_user_profile['medium_conversion_rate'], 0)
candidates['long_conversion_rate'] = np.where(candidates['is_long'] == 1, latest_user_profile['long_conversion_rate'], 0)

# Map the exact historical preferences to the candidate books
default_genre_pref = latest_user_profile['genre_preference'] if latest_user_profile['genre_preference'] > 0 else 0.5
candidates['genre_conversion_rate'] = candidates['genres'].map(genre_conv_map).fillna(0.0)
candidates['genre_preference'] = candidates['genres'].map(genre_pref_map).fillna(default_genre_pref)
candidates['author_conversion_rate'] = candidates['author_id'].map(author_conv_map).fillna(0.0)

feature_cols = [
    'overall_shelf_to_read_conversion', 'short_conversion_rate', 
    'medium_conversion_rate', 'long_conversion_rate',
    'genre_conversion_rate', 'genre_preference', 'author_conversion_rate'
]

# =======================================================
# 6. PREDICT & RANK
# =======================================================
X_candidates_scaled = scaler.transform(candidates[feature_cols])
candidates['conversion_probability'] = log_model.predict_proba(X_candidates_scaled)[:, 1]

candidates['preference_score'] = candidates['genre_preference']
X_rank = candidates[['preference_score', 'conversion_probability']]
candidates['final_rank_score'] = xgb_ranker.predict_proba(X_rank)[:, 1]

# =======================================================
# 7. DISPLAY FINAL OUTPUT
# =======================================================
top_recs = candidates.sort_values(by='final_rank_score', ascending=False).head(3)

print("=== TOP 3 PERSONALIZED RECOMMENDATIONS ===\n")
for i, (_, row) in enumerate(top_recs.iterrows(), 1):
    print(f"#{i}: {row['title']}")
    
    # Truncate raw Goodreads shelf tags so it doesn't flood the output
    raw_genre = str(row['genres'])
    clean_genre = raw_genre[:75] + "..." if len(raw_genre) > 75 else raw_genre
    print(f"    - Genre Tags: {clean_genre}")
    print(f"    - Pages: {row['num_pages']}")
    print(f"    - XGBoost Match Score: {row['final_rank_score'] * 100:.1f}%")
    print(f"    - Logistic Conversion Prob: {row['conversion_probability'] * 100:.1f}%")
    
    # Dynamic Rule-Based Explanation
    reasons = []
    if row['author_conversion_rate'] > 0:
        reasons.append("you frequently finish books by this author")
    if row['genre_conversion_rate'] > 0:
        reasons.append("this falls into a genre you highly prefer")
        
    if row['is_short'] and latest_user_profile['short_conversion_rate'] > 0.1:
        reasons.append("you complete short books reliably")
    elif row['is_medium'] and latest_user_profile['medium_conversion_rate'] > 0.1:
        reasons.append("you complete medium-length books reliably")
    elif row['is_long'] and latest_user_profile['long_conversion_rate'] > 0.1:
        reasons.append("you regularly commit to long books")
        
    reason_str = ", and ".join(reasons) if reasons else "it aligns strongly with your overall reading volume"
    print(f"    - WHY THIS BOOK? Recommended because {reason_str}.\n")

Generating optimized book recommendations for User: e259f98e50a6e241f64ed34b56b6d5b1...

=== TOP 3 PERSONALIZED RECOMMENDATIONS ===

#1: The Summer Nick Taught His Cats to Read
    - Genre Tags: ['to-read' 'picture-books' 'cats' 'picture-book' 'childrens' 'reading'
 'an...
    - Pages: 32.0
    - XGBoost Match Score: 77.9%
    - Logistic Conversion Prob: 71.0%
    - WHY THIS BOOK? Recommended because you frequently finish books by this author, and you complete short books reliably.

#2: Deenie
    - Genre Tags: ['to-read' 'young-adult' 'fiction' 'rory-gilmore-reading-challenge' 'ya'
 '...
    - Pages: 159.0
    - XGBoost Match Score: 77.9%
    - Logistic Conversion Prob: 71.0%
    - WHY THIS BOOK? Recommended because you frequently finish books by this author, and you complete short books reliably.

#3: Sir Philip's Folly  (Poor Relation, #4)
    - Genre Tags: ['to-read' 'currently-reading' 'historical-fiction' 'regency'
 'historical-...
    - Pages: 148.0
    - XGBoost Match Score: 77